# SOP: DIANN End to End Pipeline


This is a notebook for running DIA-NN workflows on ProCan's servers.

You need to run all cells but can skip sections marked as `(Optional)`.

You also need to fill out variable values in the `Manual Inputs` section before running certain cells.


In [ ]:
import logging
import os
import pprint

import pandas as pd
import procanswon
import trino
from sqlalchemy.engine import create_engine
from sqlalchemy import text

logging.basicConfig(format="%(message)s", level=logging.INFO)

### Cohort for this DIA-NN Run


**This is mandatory as it:**

    - Cross-referenced against your permissions for running DIA-NN run
    - Determines the cohort of your outputs in POCKet.**

**NOTE:** The most recent run of a cohort (engagement + project) will overwrite all others in POCKet


In [ ]:
diann_engagement = None
diann_project = None
notebook_path = os.path.abspath(".").replace(os.environ.get("HOME"), "")

### Astral / PDX Flags


In [ ]:
astral = False
pdx = False

In [ ]:
ENV = "prod"

lake = procanswon.LakeV2(
    env=ENV,
    user=os.environ.get("JUPYTERHUB_USER"),
    ec_manager_base=f"https://ec-auth-manager.{ENV}.procan.org.au",
    engagement=diann_engagement,
    project=diann_project,
    notebook=notebook_path,
    astral=astral,
    pdx=pdx,
)

In [ ]:
credentials = lake.get_pocket_credentials()
username = credentials["username"]
password = credentials["password"]
connect_args = {
    "http_scheme": "https",
    "user": username,
    "catalog": "iceberg",
    "schema": "pocket",
    "auth": trino.auth.BasicAuthentication(username, password),
}
POCKET_SQL_ALCHEMY_ENGINE = create_engine(
    f"trino://trino.{ENV}.procan.org.au:443/",
    connect_args=connect_args,
)
POCKET_SQL_ALCHEMY_CONNECTION = POCKET_SQL_ALCHEMY_ENGINE.connect()
logging.getLogger("sqlalchemy.engine").setLevel(logging.WARNING)
logging.getLogger("pandas").setLevel(logging.WARNING)

## Manual User Inputs


### Wiff Filenames


Define the `wiff_filenames` you want to build the SRL and/or search with.

**No extensions (e.g. `.wiff` or `.dia`) should be included .**


In [ ]:
wiff_filenames = [
    "210728_hek-qcs_00j8w_01vmo_m03_s_1",
    "210815_hek-qcs_021h4_01wp7_m02_s_1",
    "210814_hek-qcs_021h4_01we2_m03_s_1",
]

### Run Name


Choose a `run_name` - this is just used to name the output files you save in later cells.


In [ ]:
run_name = "a-small-example"

### Job Bookmarks


Bookmarks are used to:

    - Check the status of your job.
    - Find your job's outputs for download.
    - Specify which job's outputs you want as inputs to a new job.

E.g. when you run step 2, it prints out a bookmark ID that you record here so that it's automatically used in the next step.

e.g. `diann_srl_build_bookmark_uid = '1599f0cb-831f-40cf-8cf2-bb04c939ee44'`


In [ ]:
diann_srl_build_bookmark_uid = None  # step 2
diann_dialibqc_bookmark_uid = None  # step 2qc

diann_requant_bookmark_uid = None  # step 3a
diann_search_bookmark_uid = None  # step 3b

peptide_extraction_bookmark_uid = None  # step 4

## Step 2:


### Step 2: Build the SRL


Run the cell below to start a job to build an SRL file from the quants corresponding to your `wiff_filenames` list.

These quants are typically produced by automated Step 1 jobs (1 job per file).

If for each file, the Step 1 job:

- Hasn't been executed
- Failed
- Is still running

That will be indicated in the logs of this cell and a replacement job will be started if none are running.

**Keep rerunning this cell until no such logs appear and a bookmark for Step2 is outputted**


In [ ]:
if diann_srl_build_bookmark_uid is None:
    diann_srl_build_bookmark_obj = lake.run_diann_srl_build(wiff_filenames)  # should be a status str or a lake Artifact object
    if type(diann_srl_build_bookmark_obj) is not str:
        diann_srl_build_bookmark_uid = diann_srl_build_bookmark_obj.uid
        print(f"Running DIANN SRL build under bookmark {diann_srl_build_bookmark_uid}")
        print("Remember to save your bookmarks - i.e., replace and run the following:")
        print(f'diann_srl_build_bookmark_uid = "{diann_srl_build_bookmark_uid}"')
    else:
        print(diann_srl_build_bookmark_obj)

You can check the progress of this job at https://portal.prod.procan.org.au/#/swonbookmarks and search with the above bookmark uid.

And also run the cell below to check if the job has succeeded.


In [ ]:
if diann_srl_build_bookmark_uid is None:
    print(f"SRL hasn't been run.")
elif (
    lake.v1.get_instance_batch_status(diann_srl_build_bookmark_uid)[
        "succeeded_instance_nr"
    ]
    == 1
):
    print(f"SRL build succeeded.")
elif (
    lake.v1.get_instance_batch_status(diann_srl_build_bookmark_uid)[
        "failed_instance_nr"
    ]
    == 1
):
    print("SRL build failed!")
    pprint.pprint(lake.v1.get_logs(diann_srl_build_bookmark_uid))
else:
    print("SRL build has not finished yet!")

### Step 2: Download the SRL (Optional)


In [ ]:
diann_tsv_srl_uid = lake.v1.get_output_artifacts(
    diann_srl_build_bookmark_uid,
    output_label="diannSrlTsv",
)[0]["artifact_uid"]
# The second argument is the path of the built SRL
lake.v1.fget_artifact(diann_tsv_srl_uid, f"{run_name}-srl.tsv")

### Step 2: Run DIA Lib QC (Optional)


In [ ]:
if diann_dialibqc_bookmark_uid is None:
    diann_dialibqc_bookmark_uid = lake.run_dialib_qc(diann_srl_build_bookmark_uid).uid
    print(f"Running DIA Lib QC build under bookmark {diann_dialibqc_bookmark_uid}")
    print("Remember to save your bookmarks - i.e., replace and run the following:")
    print(f'diann_dialibqc_bookmark_uid = "{diann_dialibqc_bookmark_uid}"')

In [ ]:
if diann_dialibqc_bookmark_uid is None:
    print(f"DIA Lib QC hasn't been run.")
elif (
    lake.v1.get_instance_batch_status(diann_dialibqc_bookmark_uid)[
        "succeeded_instance_nr"
    ]
    == 1
):
    print(f"DIA Lib QC succeeded.")
elif (
    lake.v1.get_instance_batch_status(diann_dialibqc_bookmark_uid)["failed_instance_nr"]
    == 1
):
    print("DIA Lib QC failed!")
    pprint.pprint(lake.v1.get_logs(diann_dialibqc_bookmark_uid))
else:
    print("DIA Lib QC has not finished yet!")

### Step 2: Download the DIA Lib QC PDF (Optional)


In [ ]:
dialibqc_pdf_uid = lake.v1.get_output_artifacts(
    diann_dialibqc_bookmark_uid,
    output_label="pdfFile",
)[0]["artifact_uid"]

lake.v1.fget_artifact(dialibqc_pdf_uid, f"{run_name}-dialibqc.pdf")

## Step 3: Search


### Step 3: Choose wiffs and SRL


Once the SRL has been created you can run the DIA-NN search.

If you'd like to change the `wiff_filenames` for the search then do that here.


In [ ]:
# if you want to run the search on a different set of wiff filenames then do that here
# wiff_filenames = [<new filenames>]

### Step 3a: Requant the DIA Files with an SRL


In [ ]:
if diann_requant_bookmark_uid is None:
    diann_requant_bookmark_uid = lake.run_diann_search_a(
        wiff_filenames,
        diann_srl_build_bookmark_uid,
    ).uid
    print("Remember to save your bookmarks - i.e., replace and run the following:")
    print(f'diann_requant_bookmark_uid = "{diann_requant_bookmark_uid}"')

In [ ]:
if diann_requant_bookmark_uid is None:
    print(f"Requant hasn't been run.")
elif lake.v1.get_instance_batch_status(diann_requant_bookmark_uid)[
    "succeeded_instance_nr"
] == len(wiff_filenames):
    print("Requant succeeded!")
elif (
    lake.v1.get_instance_batch_status(diann_requant_bookmark_uid)["failed_instance_nr"]
    > 1
):
    print("Requant failed!")
    pprint.pprint(lake.v1.get_logs(diann_requant_bookmark_uid))
else:
    print("Requant has not finished yet!")

### Step 3b: Run Search


Now we use the bookmark uid from the previous Step3a job to run the search job.


In [ ]:
if diann_search_bookmark_uid is None:
    diann_search_bookmark_uid = lake.run_diann_search_b(diann_requant_bookmark_uid).uid
    print("Remember to save your bookmarks - i.e., replace and run the following:")
    print(f'diann_search_bookmark_uid = "{diann_search_bookmark_uid}"')

The DIANN search job should now have been kicked off. Again we can check on the progress of the DIANN search with https://portal.prod.procan.org.au/#/swonbookmarks and search with the above bookmark uid.

We can also run the cell below to check if it's succeeded.


In [ ]:
if diann_search_bookmark_uid is None:
    print(f"Search hasn't been run.")
elif (
    lake.v1.get_instance_batch_status(diann_search_bookmark_uid)[
        "succeeded_instance_nr"
    ]
    == 1
):
    print("Search succeeded!")
elif (
    lake.v1.get_instance_batch_status(diann_search_bookmark_uid)["failed_instance_nr"]
    == 1
):
    print("Search failed!")
    pprint.pprint(lake.v1.get_logs(diann_search_bookmark_uid))
else:
    print("Search has not finished yet!")

### Step 3: Download results


Now we can retrieve the output UIDs and download the files as follows:


In [ ]:
precursor_matrix_uid = lake.v1.get_output_artifacts(
    diann_search_bookmark_uid, output_label="diannPrecursorMatrixTsv"
)[0]["artifact_uid"]
protein_matrix_uid = lake.v1.get_output_artifacts(
    diann_search_bookmark_uid, output_label="diannProteinMatrixTsv"
)[0]["artifact_uid"]
gene_matrix_uid = lake.v1.get_output_artifacts(
    diann_search_bookmark_uid, output_label="diannGeneMatrixTsv"
)[0]["artifact_uid"]
uniq_gene_matrix_uid = lake.v1.get_output_artifacts(
    diann_search_bookmark_uid, output_label="diannUniqueGeneMatrixTsv"
)[0]["artifact_uid"]
logs_uid = lake.v1.get_output_artifacts(
    diann_search_bookmark_uid, output_label="diannLogTxt"
)[0]["artifact_uid"]
srl_stats_uid = lake.v1.get_output_artifacts(
    diann_search_bookmark_uid, output_label="diannSrlStatsTsv"
)[0]["artifact_uid"]
srl_quant_uid = lake.v1.get_output_artifacts(
    diann_search_bookmark_uid, output_label="diannSrlQuantTsv"
)[0]["artifact_uid"]
diann_srl_uid = lake.v1.get_output_artifacts(
    diann_srl_build_bookmark_uid, output_label="diannSrlSqlOutput"
)[0]["artifact_uid"]

In [ ]:
lake.v1.fget_artifact(precursor_matrix_uid, f"{run_name}-precursor-matrix.tsv")

In [ ]:
lake.v1.fget_artifact(protein_matrix_uid, f"{run_name}-protein-matrix.tsv")

In [ ]:
lake.v1.fget_artifact(gene_matrix_uid, f"{run_name}-gene-matrix.tsv")

In [ ]:
lake.v1.fget_artifact(uniq_gene_matrix_uid, f"{run_name}-uniq-gene-matrix.tsv")

In [ ]:
lake.v1.fget_artifact(logs_uid, f"{run_name}-logs.txt")

In [ ]:
lake.v1.fget_artifact(srl_stats_uid, f"{run_name}-search-srl-stats.tsv")

In [ ]:
lake.v1.fget_artifact(srl_quant_uid, f"{run_name}-search-srl-quant.gz")

In [ ]:
os.system(
    f"gzip -d {run_name}-search-srl-quant.gz && mv {run_name}-search-srl-quant {run_name}-search-srl-quant.tsv"
)

In [ ]:
lake.v1.fget_artifact(diann_srl_uid, f"{run_name}-srl.sql")

### Step 3: Some Resulting Metrics (Optional)


In [ ]:
pqm_matrix = pd.read_csv(
    f"{run_name}-precursor-matrix.tsv",
    sep="\t",
    usecols=["Protein.Group", "Modified.Sequence"],
)

protein_count = pd.unique(pqm_matrix["Protein.Group"]).size
peptide_count = pd.unique(pqm_matrix["Modified.Sequence"]).size

print(f"Found {protein_count} proteins and {peptide_count} modified peptides!")

## Step 4: Peptide Matrix


### Step 4: Create Peptide Matrixes


Run the peptide extraction. The bookmark uid is what you should pass onto CDS.


In [ ]:
if peptide_extraction_bookmark_uid is None:
    peptide_extraction_bookmark_uid = lake.run_diann_peptide_extraction(
        diann_search_bookmark_uid
    ).uid
    print(f"Extracting peptide matrix under bookmark {peptide_extraction_bookmark_uid}")
    print("Remember to save your bookmarks - i.e., replace and run the following:")
    print(f'peptide_extraction_bookmark_uid = "{peptide_extraction_bookmark_uid}"')

In [ ]:
if peptide_extraction_bookmark_uid is None:
    print(f"Peptide matrix extraction hasn't been run.")
elif (
    lake.v1.get_instance_batch_status(peptide_extraction_bookmark_uid)[
        "succeeded_instance_nr"
    ]
    == 1
):
    print("Peptide matrix extraction succeeded!")
elif (
    lake.v1.get_instance_batch_status(peptide_extraction_bookmark_uid)[
        "failed_instance_nr"
    ]
    == 1
):
    print("Peptide matrix extraction failed!")
    pprint.pprint(lake.v1.get_logs(peptide_extraction_bookmark_uid))
else:
    print("Peptide matrix extraction has not finished yet!")

In [ ]:
quant_uid = lake.v1.get_output_artifacts(
    peptide_extraction_bookmark_uid, output_label="diannPeptideMatrixQuantTsv"
)[0]["artifact_uid"]
lake.v1.fget_artifact(quant_uid, f"{run_name}-extracted-peptide-matrix.tsv")

In [ ]:
norm_uid = lake.v1.get_output_artifacts(
    peptide_extraction_bookmark_uid, output_label="diannPeptideMatrixNormTsv"
)[0]["artifact_uid"]
lake.v1.fget_artifact(norm_uid, f"{run_name}-extracted-peptide-matrix-norm.tsv")

## Step 5: CDS Extract (Optional)

Extract Useful ProLIMS Info about each Wiff Files from Pocket.


In [ ]:
def get_pocket_df(query: str) -> pd.DataFrame:
    return pd.read_sql(text(query), POCKET_SQL_ALCHEMY_CONNECTION)

In [ ]:
prolims_filenames = tuple(map(str.lower, wiff_filenames))
cds_extract_df = get_pocket_df(
    f"SELECT * FROM prolims WHERE ms_filename in {prolims_filenames}"
)
cds_extract_df

#### Check for Wiff Files not in Pocket

Check for files in `wiff_filenames` that haven't been successfully extracted from a ProLIMS spreadsheet.


In [ ]:
if missing_filenames := list(
    set(prolims_filenames).difference(cds_extract_df["ms_filename"].values)
):
    logging.error("The following files have no extract rows.")
    logging.error(missing_filenames)
else:
    print("All files found in Pocket")

#### Check for Wiff Files without Matching Prep Info

Each file in `wiff_filenames` should match to exactly 1 row in a Prep Tab. So files that fail to match may not have a Prep Tab with a matching LIMS ID or there may be 2 or more such rows where the match is ambiguous (e.g. pooling information may be missing).


In [ ]:
missing_prep_df = get_pocket_df(
    f"SELECT * FROM prolims WHERE ms_filename IN {prolims_filenames} AND prep_lims_id IS NULL ORDER BY ms_filename"
)
if len(missing_prep_df) > 0:
    logging.error("The following files couldn't be matched to a single Prep row")
    logging.error(list(missing_prep_df["ms_filename"]))
else:
    print("All files have matching Prep Info")

#### Export CDS Extract


In [ ]:
cds_extract_df.to_csv(f"{run_name}-cds-extract.tsv", sep="\t")